# 06 — Baselines estadísticos  (auto-generado por build_notebook_06.py)

Evalúa cuatro baselines clásicos (B0 media global, B1 persistencia naive,
B2 media móvil w∈{5,10,15}, B3 suavizado exponencial simple α=0.3) sobre los
corredores **E2** y **E59** usando el split temporal de Fase 3.

Referencia: `docs/plan-de-desarrollo.md §4 Fase 4 — Baselines estadísticos`.

In [ ]:

import polars as pl
import numpy as np
from pathlib import Path

# Locate headways parquets for E2 and E59 under /kaggle/input or local dir.
def _find_parquet(empresa_id: int) -> Path:
    name = f"headways_E{empresa_id}.parquet"
    if Path("/kaggle/input").exists():
        candidates = list(Path("/kaggle/input").rglob(name))
        if candidates:
            return candidates[0]
    candidates = list(Path(".").rglob(name))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(
        f"{name} not found. Expected at /kaggle/input/**/{name}"
    )

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)
CSV_OUT = OUTPUT_DIR / "baselines_results.csv"

print(f"Output dir: {OUTPUT_DIR}")

## Module: evaluation/splits

Temporal split helper (`split_temporal`) and train-only p99 winsorization
(`winsorize_train_p99`).  Split date ranges are locked in spec §3.

In [ ]:
"""Temporal split and winsorization helpers for headway evaluation — Fase 3.

Public API:
    split_temporal(df: pl.DataFrame) -> pl.DataFrame
    winsorize_train_p99(df: pl.DataFrame) -> tuple[pl.DataFrame, float]

Constants (split date ranges, locked in spec §3 and design §5):
    SPLIT_TRAIN_START, SPLIT_TRAIN_END
    SPLIT_VAL_START,   SPLIT_VAL_END
    SPLIT_TEST_START,  SPLIT_TEST_END
    WINSOR_QUANTILE

Design decisions (locked in design §5 and §9):
  - Split key is pl.col("t").dt.date() membership, NOT row index.
  - Three ranges are exhaustive and mutually exclusive.
  - Rows outside all three ranges receive None (split column = null).
  - Winsorization threshold is computed on train rows only (AC-WINSOR-1, AC-WINSOR-2).
  - Null delta_t_min rows are NOT clipped (AC-WINSOR-3).
  - Rows above threshold are clipped (not dropped) (AC-WINSOR-4).
  - Constants live here (not PRODUCTIVE_PARAMS) — evaluation protocol concern.
  - WINSOR_QUANTILE and split dates are not added to pyproject.toml.
"""
from __future__ import annotations

from datetime import date

import polars as pl

# ---------------------------------------------------------------------------
# Split date range constants (spec §3, inclusive on both ends)
# ---------------------------------------------------------------------------

SPLIT_TRAIN_START: date = date(2023, 10, 1)
SPLIT_TRAIN_END:   date = date(2024, 1, 15)

SPLIT_VAL_START:   date = date(2024, 1, 16)
SPLIT_VAL_END:     date = date(2024, 2, 7)

SPLIT_TEST_START:  date = date(2024, 2, 8)
SPLIT_TEST_END:    date = date(2024, 2, 29)

WINSOR_QUANTILE: float = 0.99


def split_temporal(df: pl.DataFrame) -> pl.DataFrame:
    """Add a `split` column (Utf8) with values {"train", "val", "test"}.

    Membership is determined by pl.col("t").dt.date() against the six
    module-level date constants.  Rows outside all three ranges receive
    null (should not exist in the R7 v4 dataset; harness raises if found).

    Parameters
    ----------
    df:
        headways DataFrame containing at least a `t` (Datetime) column.

    Returns
    -------
    pl.DataFrame — input frame with one added column `split: Utf8`.
    """
    day = pl.col("t").dt.date()
    return df.with_columns(
        pl.when((day >= SPLIT_TRAIN_START) & (day <= SPLIT_TRAIN_END))
          .then(pl.lit("train"))
          .when((day >= SPLIT_VAL_START) & (day <= SPLIT_VAL_END))
          .then(pl.lit("val"))
          .when((day >= SPLIT_TEST_START) & (day <= SPLIT_TEST_END))
          .then(pl.lit("test"))
          .otherwise(None)
          .alias("split")
    )


def winsorize_train_p99(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, float]:
    """Clip delta_t_min to the 99th-percentile threshold computed on train rows only.

    The threshold is computed once as a scalar from non-null train-split rows.
    It is then applied as a clip ceiling to ALL rows (train + val + test).
    Null delta_t_min values are never clipped — they remain null (AC-WINSOR-3).

    Parameters
    ----------
    df:
        headways DataFrame that already has a `split` column (added by
        split_temporal) and a `delta_t_min` (Float64 nullable) column.

    Returns
    -------
    (clipped_df, threshold)
        clipped_df: same schema as df, delta_t_min clipped.
        threshold: the scalar train-p99 value used as the clip ceiling.

    Design note (AC-WINSOR-2 leakage guard):
        The filter `split == "train"` is applied BEFORE computing the quantile,
        so extreme outliers in val or test rows cannot shift the threshold.
    """
    threshold = float(
        df.filter(
            (pl.col("split") == "train") & pl.col("delta_t_min").is_not_null()
        )["delta_t_min"]
        .quantile(WINSOR_QUANTILE)
    )

    # Clip: preserve null rows; clip non-null rows to threshold from above.
    # pl.min_horizontal(col, lit(threshold)) would coerce null → 0 in some
    # polars versions, so we use the explicit when/then pattern (design §5).
    clipped = df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
          .then(None)
          .otherwise(
              pl.min_horizontal(pl.col("delta_t_min"), pl.lit(threshold))
          )
          .alias("delta_t_min")
    )
    return clipped, threshold

## Module: evaluation/metrics

`mae` and `rmse` in minutes.  Both accept polars Series or numpy arrays.
Null/NaN rows are dropped before aggregation.  MAPE is explicitly excluded
(spec B3-NO-MAPE).

In [ ]:
"""Evaluation metrics for headway forecasting — Fase 3.

Public API:
    mae(y_true, y_pred) -> float
    rmse(y_true, y_pred) -> float

Both functions accept polars Series (Float64) or numpy arrays (float64).
Null / NaN masking: rows where EITHER y_true or y_pred is null/NaN are
dropped before aggregation.  If no valid rows remain, ValueError is raised.

Design decisions locked in design §4:
  - ValueError on empty/all-null input (NOT silent NaN return).
  - Only MAE and RMSE are in scope (spec B3-NO-MAPE — ratio-based metrics
    are out of scope because near-zero headways cause denominator blow-up).
  - No new pyproject.toml dependencies (polars + numpy already present).
"""
from __future__ import annotations

import numpy as np
import polars as pl


def _to_numpy_with_mask(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Coerce both inputs to float64 numpy arrays and apply the null/NaN mask.

    Polars Series with dtype Float64: null cells become NaN via .to_numpy().
    numpy arrays: assumed to already use NaN for missing values.

    Returns
    -------
    (y_true_masked, y_pred_masked) — two 1-D float64 arrays of equal length,
    containing no NaN values.  May be empty if all rows were masked.
    """
    # Coerce to numpy.
    if isinstance(y_true, pl.Series):
        yt = y_true.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yt = np.asarray(y_true, dtype=np.float64).ravel()

    if isinstance(y_pred, pl.Series):
        yp = y_pred.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yp = np.asarray(y_pred, dtype=np.float64).ravel()

    # Elementwise mask: keep row only if BOTH sides are finite (not NaN).
    mask = ~(np.isnan(yt) | np.isnan(yp))
    return yt[mask], yp[mask]


def mae(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Mean Absolute Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — MAE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "mae: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.mean(np.abs(yt - yp)))


def rmse(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Root Mean Squared Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — RMSE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "rmse: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.sqrt(np.mean((yt - yp) ** 2)))

## Module: baselines/statistical

B0 global mean, B1 naive persistence, B2 moving average (w∈{5,10,15}),
B3 simple exponential smoothing (α=0.3).  All operate per slot
`(empresaid, direction, pair_rank)`.

In [ ]:
"""Classical statistical baselines for headway forecasting — Fase 3.

Public API:
    predict_b0(headways: pl.DataFrame) -> pl.DataFrame
    predict_b1(headways: pl.DataFrame) -> pl.DataFrame
    predict_b2(headways: pl.DataFrame, *, window: int) -> pl.DataFrame
    predict_b3(headways: pl.DataFrame, *, alpha: float = SES_ALPHA) -> pl.DataFrame

Input contract (all four functions):
    The DataFrame must have a `split` column (Utf8) added by split_temporal.
    Columns consumed: empresaid, t, direction, pair_rank, delta_t_min, split.

Output contract:
    Each function returns the input frame with ONE additional column:
        B0 → y_pred_b0
        B1 → y_pred_b1
        B2 → y_pred_b2_w{window}
        B3 → y_pred_b3

Predictions are filled for ALL rows (train + test); the evaluation harness
consumes test rows only.  Filling train rows costs negligibly more and lets
future SDDs reuse predictions if needed (design §3).

Design decisions locked in design §3 and §9:
  - Functions, not classes (consistent with project precedent).
  - Slot key: (empresaid, direction, pair_rank).
  - B2 `window` = count of last NON-NULL observations (not a time window).
  - B2 min_periods = window // 2  (floor division).
  - B3 alpha = SES_ALPHA = 0.3, per-slot online recursion, null-skip.
  - B3 state init: NaN until first non-null train obs; first non-null sets s directly.
  - No new pyproject.toml dependencies (polars + numpy only).
"""
from __future__ import annotations

import numpy as np
import polars as pl

# ---------------------------------------------------------------------------
# Module-level constants (locked in design §9)
# ---------------------------------------------------------------------------

BASELINE_B2_WINDOWS: tuple[int, ...] = (5, 10, 15)
SES_ALPHA: float = 0.3

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


# ===========================================================================
# B0 — Global mean per slot (train rows only)
# ===========================================================================

def predict_b0(headways: pl.DataFrame) -> pl.DataFrame:
    """Add column `y_pred_b0`: per-slot mean of train delta_t_min.

    The prediction is constant within a slot — the arithmetic mean of all
    non-null delta_t_min values in the train split for that slot.  Slots
    with no non-null train observations receive null (AC-B0-2).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b0` (Float64 nullable) added.
    """
    train_means = (
        headways
        .filter(pl.col("split") == "train")
        .group_by(_SLOT_COLS)
        .agg(pl.col("delta_t_min").mean().alias("y_pred_b0"))
    )
    return headways.join(train_means, on=_SLOT_COLS, how="left")


# ===========================================================================
# B1 — Naive / persistence baseline
# ===========================================================================

def predict_b1(headways: pl.DataFrame) -> pl.DataFrame:
    """Add column `y_pred_b1`: last non-null delta_t_min seen before each row.

    Uses forward_fill().shift(1).over(slot) — the canonical polars pattern for
    ŷ_{t+1} = y_t with null gaps.  Causal by construction (shift prevents
    the current-row value from appearing as its own prediction).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.

    Returns
    -------
    pl.DataFrame — input frame sorted by (slot, t), with `y_pred_b1` added.
    """
    return (
        headways
        .sort(_SLOT_COLS + ["t"])
        .with_columns(
            pl.col("delta_t_min")
              .forward_fill()
              .shift(1)
              .over(_SLOT_COLS)
              .alias("y_pred_b1")
        )
    )


# ===========================================================================
# B2 — Trailing moving average of last w NON-NULL observations
# ===========================================================================

def predict_b2(headways: pl.DataFrame, *, window: int) -> pl.DataFrame:
    """Add column `y_pred_b2_w{window}`: mean of last `window` non-null observations.

    Semantics (locked, design §3):
      - `window` is a COUNT of non-null observations, not a time window.
      - min_periods = window // 2  (floor).
      - Prediction at row i uses only observations strictly before row i (causal).

    Implementation:
      - group_by(slot).map_groups(lambda g: _b2_one_slot(g, window))
      - Within each group: extract non-null values, compute rolling_mean with
        shift(1) (causal), then join_asof(strategy="backward") back to
        original group rows on `t`.

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    window:
        Number of non-null observations in the trailing window.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b2_w{window}` (Float64 nullable) added.
    """
    col_name = f"y_pred_b2_w{window}"
    min_periods = window // 2

    def _b2_one_slot(group: pl.DataFrame) -> pl.DataFrame:
        group = group.sort("t")

        # Extract non-null rows only, in time order.
        non_null = group.filter(pl.col("delta_t_min").is_not_null())

        if len(non_null) == 0:
            # No non-null observations: all predictions are null.
            return group.with_columns(pl.lit(None, dtype=pl.Float64).alias(col_name))

        # Compute rolling mean on the non-null sub-series, then shift(1) for
        # causality: the prediction at position i uses observations 0..i-1.
        non_null = non_null.with_columns(
            pl.col("delta_t_min")
              .rolling_mean(window_size=window, min_samples=min_periods)
              .shift(1)
              .alias(col_name)
        )

        # Align back to the full group (including null rows) via join_asof.
        # strategy="backward" finds the most recent non-null rolling mean at or
        # before each timestamp in the original group.
        result = group.join_asof(
            non_null.select(["t", col_name]),
            on="t",
            strategy="backward",
        )
        return result

    result = (
        headways
        .sort(_SLOT_COLS + ["t"])
        .group_by(_SLOT_COLS, maintain_order=True)
        .map_groups(_b2_one_slot)
    )
    return result


# ===========================================================================
# B3 — Simple Exponential Smoothing (α=0.3, per slot, online)
# ===========================================================================

def _ses_one_slot(slot_df: pl.DataFrame, alpha: float) -> pl.DataFrame:
    """Online SES recursion for a single slot.

    s_t = α·y_t + (1-α)·s_{t-1}  (null observations skip the update).
    pred[i] = s before observing y[i]  (causal: shift-1 semantics).

    Initialization: s = NaN until the first non-null y_t; the first non-null
    value sets s directly (no prior needed) — AC-B3-3.  The prediction at
    that initialization row is NaN (no prior state), so the first test
    prediction for a slot with at least one train observation is the state
    after consuming ALL train rows.
    """
    slot_df = slot_df.sort("t")
    y = slot_df["delta_t_min"].to_numpy(allow_copy=True).astype(np.float64)
    pred = np.full(len(y), np.nan)
    s = np.nan  # smoothing state; NaN until first non-null

    for i in range(len(y)):
        # Prediction at row i is the state BEFORE observing y[i].
        pred[i] = s
        # Update state if current observation is not null/NaN.
        if not np.isnan(y[i]):
            if np.isnan(s):
                s = y[i]  # initialization: first non-null sets state directly
            else:
                s = alpha * y[i] + (1.0 - alpha) * s

    # Convert float NaN → polars null so downstream is_null() works correctly.
    pred_series = pl.Series("y_pred_b3", pred, dtype=pl.Float64)
    return slot_df.with_columns(pred_series.set(pred_series.is_nan(), None))


def predict_b3(headways: pl.DataFrame, *, alpha: float = SES_ALPHA) -> pl.DataFrame:
    """Add column `y_pred_b3`: online SES predictions, α=0.3 (default).

    Applies the causal recursion s_t = α·y_t + (1-α)·s_{t-1} per slot.
    Null observations do not update the state (AC-B3-2).
    State is initialized from the first non-null observation (AC-B3-3).
    Slots with all-null values emit null for all rows (AC-B3-4).

    Parameters
    ----------
    headways:
        headways DataFrame with `split` column attached.
    alpha:
        Smoothing parameter.  Default SES_ALPHA = 0.3 (locked, design §3).
        Tests may pass alternative values for edge-case verification.

    Returns
    -------
    pl.DataFrame — input frame with `y_pred_b3` (Float64 nullable) added.

    Design note (D-PL-OVER-VS-MAPGROUPS):
        polars .over() does not support stateful numpy loops; map_groups
        materializes one Python frame per slot (~30–100 per corridor — trivially fast).
    """
    return (
        headways
        .sort(_SLOT_COLS + ["t"])
        .group_by(_SLOT_COLS, maintain_order=True)
        .map_groups(lambda g: _ses_one_slot(g, alpha))
    )

## Module: baselines/harness

`evaluate_corridor` composes split → winsorize → all baselines → metrics per
(direction × baseline) and returns a tidy 36-row long DataFrame.

In [ ]:
"""Evaluation harness for classical baseline comparison — Fase 3.

Public API:
    evaluate_corridor(headways: pl.DataFrame, corridor_name: str) -> pl.DataFrame

The function composes the full pipeline for one corridor:
    split_temporal → winsorize_train_p99 → predict_b0/b1/b2(×3)/b3
    → filter test rows → compute MAE + RMSE per (direction, baseline)
    → return tidy long-form DataFrame.

Output schema (design §6):
    corridor   Utf8
    direction  Utf8   — "-1", "+1", "aggregate"
    baseline   Utf8   — "B0", "B1", "B2_w5", "B2_w10", "B2_w15", "B3"
    metric     Utf8   — "MAE", "RMSE"
    value      Float64 — minutes

Rows per corridor: 3 directions × 6 baselines × 2 metrics = 36.
Both corridors together (notebook caller): 72 rows.

Design decisions (locked in design §6 and §9):
  - "aggregate" direction = MAE/RMSE over POOLED test rows (both directions
    concatenated), NOT mean of per-direction metrics.
  - val rows are NEVER consumed (B3-VAL-UNUSED).
  - No new pyproject.toml dependencies.
  - harness.py does NOT read parquets or write CSV (notebook does those).
"""
from __future__ import annotations

import polars as pl

from src.evaluation.metrics import mae, rmse
from src.evaluation.splits import split_temporal, winsorize_train_p99
from src.baselines.statistical import (
    BASELINE_B2_WINDOWS,
    predict_b0,
    predict_b1,
    predict_b2,
    predict_b3,
)

# Map from prediction column name → display name for the output DataFrame.
_BASELINE_MAP: list[tuple[str, str]] = [
    ("y_pred_b0", "B0"),
    ("y_pred_b1", "B1"),
    ("y_pred_b2_w5", "B2_w5"),
    ("y_pred_b2_w10", "B2_w10"),
    ("y_pred_b2_w15", "B2_w15"),
    ("y_pred_b3", "B3"),
]


def evaluate_corridor(
    headways: pl.DataFrame,
    corridor_name: str,
) -> pl.DataFrame:
    """Run all classical baselines on one corridor and return a tidy metrics table.

    Parameters
    ----------
    headways:
        Raw headways DataFrame with R7 v4 schema columns:
        empresaid, t, direction, pair_rank, delta_t_min.
        Must NOT already have a `split` column (this function adds it).
    corridor_name:
        Label for the `corridor` column in the output (e.g. "E2", "E59").

    Returns
    -------
    pl.DataFrame — 36-row tidy long-form table with schema:
        [corridor, direction, baseline, metric, value]

    Notes
    -----
    - val rows are ignored at prediction time (baselines consume train only)
      and are never included in metric computation (metrics use test rows only).
    - The "aggregate" direction row pools test rows from both directions before
      computing MAE/RMSE — it is NOT the mean of the two per-direction metrics.
    """
    # --- Pipeline: split → winsorize → all baselines ---
    df = split_temporal(headways)
    df, _threshold = winsorize_train_p99(df)

    df = predict_b0(df)
    df = predict_b1(df)
    for w in BASELINE_B2_WINDOWS:
        df = predict_b2(df, window=w)
    df = predict_b3(df)

    # --- Filter to test rows only (B3-VAL-UNUSED) ---
    test_df = df.filter(pl.col("split") == "test")

    # --- Compute metrics per (direction × baseline) ---
    rows: list[dict] = []

    for pred_col, baseline_name in _BASELINE_MAP:
        for direction_val in (-1, 1, "aggregate"):
            if direction_val == "aggregate":
                # Pool all test rows regardless of direction.
                subset = test_df
                direction_str = "aggregate"
            else:
                subset = test_df.filter(pl.col("direction") == direction_val)
                direction_str = f"+{direction_val}" if direction_val > 0 else str(direction_val)

            y_true = subset["delta_t_min"]
            y_pred = subset[pred_col]

            # Compute MAE and RMSE (null rows are masked inside the functions).
            mae_val = mae(y_true, y_pred)
            rmse_val = rmse(y_true, y_pred)

            rows.append(
                {
                    "corridor": corridor_name,
                    "direction": direction_str,
                    "baseline": baseline_name,
                    "metric": "MAE",
                    "value": mae_val,
                }
            )
            rows.append(
                {
                    "corridor": corridor_name,
                    "direction": direction_str,
                    "baseline": baseline_name,
                    "metric": "RMSE",
                    "value": rmse_val,
                }
            )

    return pl.DataFrame(rows).with_columns(
        pl.col("corridor").cast(pl.Utf8),
        pl.col("direction").cast(pl.Utf8),
        pl.col("baseline").cast(pl.Utf8),
        pl.col("metric").cast(pl.Utf8),
        pl.col("value").cast(pl.Float64),
    )

## Cargar datos — E2 y E59

Lee los parquets v5 generados por el notebook 04 (two-pass pipeline).

In [ ]:

hw_e2  = pl.read_parquet(_find_parquet(2))
hw_e59 = pl.read_parquet(_find_parquet(59))

print(f"E2:  {hw_e2.height:,} rows, {hw_e2.width} cols")
print(f"E59: {hw_e59.height:,} rows, {hw_e59.width} cols")

## Sanidad — cobertura por (empresa, dirección)

Fracción de `delta_t_min` no-nulo y conteo de filas por dirección.
Permite detectar baja cobertura en dir=+1 (riesgo R-DIR1-COVERAGE).

In [ ]:

for label, hw in [("E2", hw_e2), ("E59", hw_e59)]:
    print(f"\n=== {label} ===")
    summary = (
        hw.with_columns(pl.col("t").dt.date().alias("day"))
          .group_by(["empresaid", "direction"])
          .agg([
              pl.len().alias("n_rows"),
              (pl.col("delta_t_min").is_not_null().sum() / pl.len()).alias("non_null_frac"),
          ])
          .sort(["empresaid", "direction"])
    )
    print(summary)

## Ejecutar harness — split → winsorize → baselines → métricas

Llama a `evaluate_corridor` para E2 y E59 y concatena los resultados.

In [ ]:

results_e2  = evaluate_corridor(hw_e2,  "E2")
results_e59 = evaluate_corridor(hw_e59, "E59")
results = pl.concat([results_e2, results_e59])

print(f"Total rows: {results.height}  (expected 72 = 2 corridors × 36)")
print(results.head(10))

## Escribir CSV — baselines_results.csv

Escribe la tabla long-form a `/kaggle/working/baselines_results.csv`.
Columnas: `corridor, direction, baseline, metric, value`.

In [ ]:

results.write_csv(CSV_OUT)
print(f"CSV written to: {CSV_OUT}")
print(f"Rows: {results.height}  Columns: {results.columns}")

## Tabla resumen (wide format)

Pivote ancho para lectura humana: filas = (corridor, direction),
columnas = (baseline, metric).

In [ ]:

wide = results.pivot(
    on="baseline",
    index=["corridor", "direction", "metric"],
    values="value",
)
print(wide.sort(["corridor", "direction", "metric"]))